<a href="https://colab.research.google.com/github/varba187/RAGs-to-Riches/blob/main/code/notebooks/dpr_eval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip -q install transformers "datasets<3" sentencepiece accelerate faiss-cpu

In [ ]:
import re
import torch
import faiss
import numpy as np
import pandas as pd

from datasets import load_dataset
from transformers import (
    DPRQuestionEncoder,
    DPRQuestionEncoderTokenizer,
    DPRContextEncoder,
    DPRContextEncoderTokenizer,
    BartTokenizer,
    BartForConditionalGeneration
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device set to {device}")

torch.manual_seed(0)

# NQ dataset

In [ ]:
nq = load_dataset("sentence-transformers/natural-questions", split="train[:500]")
dataset_name = "nq"

# Helper functions

In [ ]:
def get_question(example):
  return example["query"]

def get_answer(example):
  answer = example["answer"]
  if isinstance(answer, list):
    answer = answer[0]
  return answer

In [ ]:
def normalize_text(text):
    text = str(text).lower().strip()
    text = re.sub(r"\s+", " ", text)
    return text

# Build Passage Corpus

In [ ]:
def build_passage_corpus(dataset, answer_field_fn):
  passages = []
  for i in range(len(dataset)):
    passages.append(answer_field_fn(dataset[i]))
  passages = list(dict.fromkeys(passages))
  return passages

passages = build_passage_corpus(nq, get_answer)
print(f"Number of passages: {len(passages)}")
print(passages[0])

# Load DPR models

In [ ]:
question_tokenizer = DPRQuestionEncoderTokenizer.from_pretrained("facebook/dpr-question_encoder-single-nq-base")
question_encoder = DPRQuestionEncoder.from_pretrained("facebook/dpr-question_encoder-single-nq-base")
question_encoder = question_encoder.to(device)


context_tokenizer = DPRContextEncoderTokenizer.from_pretrained("facebook/dpr-ctx_encoder-single-nq-base")
context_encoder = DPRContextEncoder.from_pretrained("facebook/dpr-ctx_encoder-single-nq-base")
context_encoder = context_encoder.to(device)


bart_tokenizer = BartTokenizer.from_pretrained("facebook/bart-large")
bart_model = BartForConditionalGeneration.from_pretrained("facebook/bart-large")
bart_model = bart_model.to(device)

# Question Encoding

In [ ]:
def encode_question(question):
  inputs = question_tokenizer(question, return_tensors="pt", truncation=True, padding=True, max_length=128)
  inputs = {k: v.to(device) for k, v in inputs.items()}
  with torch.no_grad():
    outputs = question_encoder(**inputs).pooler_output
  return outputs.squeeze(0).cpu().numpy().astype(np.float32)

# Passage Encoding

In [ ]:
def encode_passage(passage):
  inputs = context_tokenizer(passage, return_tensors="pt", truncation=True, padding=True, max_length=128)
  inputs = {k: v.to(device) for k, v in inputs.items()}

  with torch.no_grad():
    outputs = context_encoder(**inputs).pooler_output

  return outputs.squeeze(0).cpu().numpy().astype(np.float32)

# Build FAISS index

In [ ]:
def build_faiss_index(passages):
  passages_embeddings = np.stack([encode_passage(passage) for passage in passages]).astype(np.float32)
  faiss.normalize_L2(passages_embeddings)
  index = faiss.IndexFlatIP(passages_embeddings.shape[1])
  index.add(passages_embeddings)
  return index

index = build_faiss_index(passages)

# Retrieval function

In [ ]:
def retrieval_top_k(question, index, passages, k = 5):
  question_embedding = encode_question(question)
  question_embedding = question_embedding.reshape(1, -1)
  faiss.normalize_L2(question_embedding)
  distances, indices = index.search(question_embedding, k)
  relevant_passages = [passages[i] for i in indices[0]]
  return relevant_passages, distances[0]

# Prediction rule

In [ ]:
def dpr_prediction_rule(question, index, passages, k = 5):
  relevant_passages, distances = retrieval_top_k(question, index, passages, k)
  prediction = relevant_passages[0]
  return prediction, relevant_passages

# Smoke Test

In [ ]:
question = get_question(nq[0])
answer = get_answer(nq[0])

prediction, relevant_passages = dpr_prediction_rule(question, index, passages)

print("Question: ", question)
print("Answer: ", answer)
print("Prediction: ", prediction)
print("Relevant passages: ", relevant_passages)

# Evaluation

In [ ]:
results = []

for i in range(50):
  question = get_question(nq[i])
  answer = get_answer(nq[i])
  prediction, relevant_passages = dpr_prediction_rule(question, index, passages)
  em = int(normalize_text(answer) == normalize_text(prediction))
  results.append({
        "dataset": dataset_name,
        "question_number": i,
        "question": question,
        "answer": answer,
        "prediction": prediction,
        "em": em,
        "relevant_passages": relevant_passages
    })

results_df = pd.DataFrame(results)
results_df.head()

In [ ]:
final_em = results_df["em"].mean()*100
print("DPR EM: ", final_em)
results_df.to_csv("dpr_results.csv", index=False)
print("Saved dpr_results.csv")